In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Create train and test DataFrame 

In [ ]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

train.drop(columns='Id', inplace=True)

In [ ]:
train.describe

Preprocessing

Fill nan and one hot encode

In [ ]:
numeric_cols = train.select_dtypes(include=['int64','float64']).columns
categorical_cols = train.select_dtypes(include=['object']).columns

train = train.dropna(subset=['SalePrice'])
numeric_cols = numeric_cols.drop('SalePrice', errors='ignore')

# Fill numeric
for col in numeric_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

# Fill categorical
for col in categorical_cols:
    train[col] = train[col].fillna('Missing')
    test[col] = test[col].fillna('Missing')

# One-hot encode
y_train = train['SalePrice']
train = train.drop('SalePrice', axis=1)
n_train = len(train)

test_ids = test['Id'] 
test = test.drop('Id', axis=1) 

combined_df = pd.concat([train, test], axis=0)
combined_df_encoded = pd.get_dummies(combined_df, columns=categorical_cols, drop_first=True)

In [ ]:
combined_df_encoded = combined_df_encoded.fillna(0) 
X_train = combined_df_encoded.iloc[:n_train, :]
X_test = combined_df_encoded.iloc[n_train:, :]

In [ ]:
X_train.head()

In [ ]:
y_train.describe

In [ ]:
X_train.isna().any(axis=1).sum()

Scale numeric columns

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train[numeric_cols])

X_train.loc[:, numeric_cols] = scaler.transform(X_train[numeric_cols])
X_test.loc[:, numeric_cols] = scaler.transform(X_test[numeric_cols])

y_train = np.log1p(y_train)
y_train_reshaped = y_train.values.reshape(-1, 1)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train_reshaped)
y_train = y_train_scaled

Train test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train_new, X_val, y_train_new, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

print(f"Full X_train shape: {X_train.shape}")
print(f"New X_train_new shape: {X_train_new.shape}")
print(f"Validation X_val shape: {X_val.shape}")

Build simple multi layer perceptron weights and bias

In [ ]:
np.random.seed(42)
n_input  = X_train.shape[1]
n_layer0 = 32
n_layer1 = 64
n_layer2 = 128
n_layer3 = 256
n_output = 1

W0 = np.random.randn(n_input,  n_layer0) * np.sqrt(2.0/n_input)
b0 = np.zeros((1, n_layer0))
W1 = np.random.randn(n_layer0,  n_layer1) * np.sqrt(2.0/n_layer0)
b1 = np.zeros((1, n_layer1))
W2 = np.random.randn(n_layer1, n_layer2) * np.sqrt(2.0/n_layer1)
b2 = np.zeros((1, n_layer2))
W3 = np.random.randn(n_layer2, n_layer3) * np.sqrt(2.0/n_layer2)
b3 = np.zeros((1, n_layer3))
W4 = np.random.randn(n_layer3, n_output) * np.sqrt(2.0/n_layer3)
b4 = np.zeros((1, n_output))

W0 = W0.astype(float)
b0 = b0.astype(float)
W1 = W1.astype(float)
b1 = b1.astype(float)
W2 = W2.astype(float)
b2 = b2.astype(float)
W3 = W3.astype(float)
b3 = b3.astype(float)
W4 = W4.astype(float)
b4 = b4.astype(float)

Simple MLP model

In [ ]:
dropout_rate = 0.05
keep_prob = 1 - dropout_rate

def ReLU(Y):
    return np.maximum(0, Y)

def ReLU_derivative(Y):
    return (Y > 0).astype(float)

def forward(X, is_training=True):
    Y0 = X.dot(W0) + b0
    A0 = ReLU(Y0)
    D0 = None # Khởi tạo mask
    if is_training:
        D0 = (np.random.rand(*A0.shape) < keep_prob) / keep_prob
        A0 = A0 * D0
        
    Y1 = A0.dot(W1) + b1
    A1 = ReLU(Y1)
    D1 = None 
    if is_training:
        D1 = (np.random.rand(*A1.shape) < keep_prob) / keep_prob
        A1 = A1 * D1

    Y2 = A1.dot(W2) + b2
    A2 = ReLU(Y2)
    D2 = None
    if is_training:
        D2 = (np.random.rand(*A2.shape) < keep_prob) / keep_prob
        A2 = A2 * D2

    Y3 = A2.dot(W3) + b3
    A3 = ReLU(Y3)
    D3 = None
    if is_training:
        D3 = (np.random.rand(*A3.shape) < keep_prob) / keep_prob
        A3 = A3 * D3

    y_pred = A3.dot(W4) + b4
    
    return Y0, A0, D0, Y1, A1, D1, Y2, A2, D2, Y3, A3, D3, y_pred

def mse_loss(y_pred, y_train):
    m = y_pred.shape[0]
    return np.sum((y_train - y_pred)**2) / m

In [ ]:
def backward(X, y, Y0, A0, D0, Y1, A1, D1, Y2, A2, D2, Y3, A3, D3, y_pred, lr = 0.0001, weight_decay = 1e-4):
    global W0, b0, W1, b1, W2, b2, W3, b3, W4, b4

    m = X.shape[0]
    dLoss = 2 * (y_pred - y) / m

    dW4 = A3.T.dot(dLoss)
    db4 = np.sum(dLoss, axis=0, keepdims=True)
    
    dA3 = dLoss.dot(W4.T)
    dA3 = dA3 * D3 # apply mask dropout
    dy3 = ReLU_derivative(Y3) * dA3
    dW3 = A2.T.dot(dy3)
    db3 = np.sum(dy3, axis=0, keepdims=True)

    dA2 = dy3.dot(W3.T)
    dA2 = dA2 * D2 # apply mask dropout
    dy2 = ReLU_derivative(Y2) * dA2
    dW2 = A1.T.dot(dy2)
    db2 = np.sum(dy2, axis=0, keepdims=True)

    dA1 = dy2.dot(W2.T)
    dA1 = dA1 * D1 # <-- apply mask dropout
    dy1 = ReLU_derivative(Y1) * dA1
    dW1 = A0.T.dot(dy1)
    db1 = np.sum(dy1, axis=0, keepdims=True)

    dA0 = dy1.dot(W1.T)
    dA0 = dA0 * D0 # <-- apply mask dropout
    dy0 = ReLU_derivative(Y0) * dA0
    dW0 = X.T.dot(dy0)
    db0 = np.sum(dy0, axis=0, keepdims=True)

    W0 -= lr * (dW0 + 2 * weight_decay * W0)
    b0 -= lr * db0
    W1 -= lr * (dW1 + 2 * weight_decay * W1)
    b1 -= lr * db1
    W2 -= lr * (dW2 + 2 * weight_decay * W2)
    b2 -= lr * db2
    W3 -= lr * (dW3 + 2 * weight_decay * W3)
    b3 -= lr * db3
    W4 -= lr * (dW4 + 2 * weight_decay * W4)
    b4 -= lr * db4

Training 

In [ ]:
batch_size = 32
epochs = 2700

if isinstance(X_train_new, pd.DataFrame):
    X = X_train_new.values
else:
    X = X_train_new
X = np.array(X, dtype=float)
y = y_train_new 

if isinstance(X_val, pd.DataFrame):
    X_v = X_val.values
else:
    X_v = X_val
X_v = np.array(X_v, dtype=float)
y_v = y_val 

m = X.shape[0] 

for epoch in range(epochs):
    permutation = np.random.permutation(m)
    X_shuffled = X[permutation]
    y_shuffled = y[permutation]

    for i in range(0, m, batch_size):
        # Lấy ra mini-batch
        X_batch = X_shuffled[i : i + batch_size]
        y_batch = y_shuffled[i : i + batch_size]

        # Forward pass
        Y0, A0, D0, Y1, A1, D1, Y2, A2, D2, Y3, A3, D3, y_pred_batch = forward(X_batch, is_training=True)

        # Backward pass
        backward(X_batch, y_batch, Y0, A0, D0, Y1, A1, D1, Y2, A2, D2, Y3, A3, D3, y_pred_batch)
        
    if epoch % 100 == 99 or epoch == 0:
        _, _, _, _, _, _, _, _, _, _, _, _, y_pred_train = forward(X, is_training=False)
        train_loss = mse_loss(y_pred_train, y)
        train_ss_res = np.sum((y - y_pred_train) ** 2)
        train_ss_tot = np.sum((y - np.mean(y)) ** 2)
        train_r2 = 1 - (train_ss_res / train_ss_tot) if train_ss_tot > 0 else 0

        _, _, _, _, _, _, _, _, _, _, _, _, y_pred_val = forward(X_v, is_training=False)
        val_loss = mse_loss(y_pred_val, y_v)
        val_ss_res = np.sum((y_v - y_pred_val) ** 2)
        val_ss_tot = np.sum((y_v - np.mean(y_v)) ** 2)
        val_r2 = 1 - (val_ss_res / val_ss_tot) if val_ss_tot > 0 else 0

        print(f"Epoch {epoch + 1}:")
        print(f"  TRAIN: Loss = {train_loss:.4f}, R² = {train_r2:.4f}")
        print(f"  VALID: Loss = {val_loss:.4f}, R² = {val_r2:.4f}") 

Predict on test data set and create submission

In [ ]:
if isinstance(X_test, pd.DataFrame):
    X_t = X_test.values
else:
    X_t = X_test

X_t = np.array(X_t, dtype=float)
_, _, _, _, _, _, _, _, _, _, _, _, y_test_pred_scaled = forward(X_t, is_training=False)

y_test_pred_log = y_scaler.inverse_transform(y_test_pred_scaled)
y_test_pred_final = np.expm1(y_test_pred_log.flatten())

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': y_test_pred_final
})
submission.to_csv('submission.csv', index=False)